# RDLS v0.3 Metadata Validator

**Purpose**: Validate **any** RDLS metadata JSON file (manually created, editor-generated,
or pipeline output) against the official schema, report all compliance issues, and
produce revised `*_rev.json` files with auto-fixable issues corrected.

**Proof of concept** for the `to-rdls` metadata toolkit.

### Input modes
| Mode | Description |
|:------|:-------------|
| **Single file** | One JSON path |
| **Folder** | All `*.json` in a directory |
| **File list** | Explicit list of paths |

### What it checks
1. **JSON Schema validation** (Draft 2020-12) — structural + codelist compliance
2. **RDLS business rules** — required attribution roles, resource URLs, component consistency
3. **Empty-value cleanup** — `""`, `{}`, `[]` in optional fields
4. **Type coercion** — string numbers → int/float where schema expects numeric
5. **Codelist correction** — fuzzy match invalid values to nearest valid codelist entry
6. **Structural inference** — rebuild missing required fields from parent/sibling context

### Outputs
- Per-file validation report (error count, error list by category)
- Summary table across all files
- Revised `*_rev.json` files with auto-fixable issues corrected
- Comparison table: original vs revised

### Design
- **Schema-driven** — all codelist lookups and required fields come from the JSON schema
- **No hardcoded error patterns** — works with any valid or invalid RDLS record
- **Non-destructive** — original files are never modified; revisions saved as `*_rev.json`

## Cell 1 — Setup & Imports

In [12]:
import sys, json, re, copy
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime

# ---------------------------------------------------------------------------
# Path setup
# ---------------------------------------------------------------------------
TORDLS_ROOT = Path("../").resolve()          # to-rdls/
sys.path.insert(0, str(TORDLS_ROOT))

from src.utils import load_json, write_json, load_yaml, iter_json_files
from src.schema import load_rdls_schema, load_codelists, SchemaContext
from src.validate_qa import (
    validate_against_schema,
    check_business_rules,
    AutoFixer,
    validate_and_score,
    compute_composite_confidence,
)

# jsonschema (also used directly by validate_against_schema)
try:
    from jsonschema import Draft202012Validator
    print(f"jsonschema loaded (Draft202012Validator)")
except ImportError:
    raise ImportError("Install jsonschema: pip install jsonschema")

print(f"to-rdls root: {TORDLS_ROOT}")
print(f"Ready.")


jsonschema loaded (Draft202012Validator)
to-rdls root: C:\Users\benny\OneDrive\Documents\Github\hdx-metadata-crawler\to-rdls
Ready.


## Cell 2 — Configuration

In [13]:
# =============================================================================
# CONFIGURATION — Edit this cell to select input files
# =============================================================================

# Project root (parent of to-rdls/)
PROJECT_ROOT = TORDLS_ROOT.parent

# Schema path (golden reference — never edit)
SCHEMA_PATH = PROJECT_ROOT / "hdx_dataset_metadata_dump" / "rdls" / "schema" / "rdls_schema_v0.3.json"

# Schema YAML config (for codelist reference)
SCHEMA_YAML = TORDLS_ROOT / "configs" / "rdls_schema.yaml"

# Defaults YAML config
DEFAULTS_YAML = TORDLS_ROOT / "configs" / "rdls_defaults.yaml"

# ---------------------------------------------------------------------------
# INPUT MODE: Choose ONE of three modes
# ---------------------------------------------------------------------------
#   MODE = "file"    → validate a single file
#   MODE = "folder"  → validate all *.json in a folder
#   MODE = "list"    → validate specific files from a list
# ---------------------------------------------------------------------------
MODE = "folder"  # <-- Change as needed

# --- Mode: "file" — single file path ---
INPUT_FILE = PROJECT_ROOT / "hdx_dataset_metadata_dump" / "rdls" / "example" / "rdls_hevl-bgdtmrwcities_chattogram.json"

# --- Mode: "folder" — all JSON files in this directory ---
# INPUT_FOLDER = PROJECT_ROOT / "hdx_dataset_metadata_dump" / "rdls" / "example"
INPUT_FOLDER = TORDLS_ROOT / "output" / "tmrwcities" / "template"

# --- Mode: "list" — explicit list of file paths ---
INPUT_LIST = [
    PROJECT_ROOT / "hdx_dataset_metadata_dump" / "rdls" / "example" / "rdls_hevl-bgdtmrwcities_chattogram.json",
    # Add more paths here...
]

# ---------------------------------------------------------------------------
# OUTPUT
# ---------------------------------------------------------------------------
# Save revised files? If True, creates *_rev.json for files with auto-fixes
SAVE_REVISED = True

# Output folder for revised files (None = same folder as input, with _rev suffix)
REVISED_OUTPUT_DIR = None  # e.g., PROJECT_ROOT / "rdls" / "revised"

# Skip files ending with _rev.json? (avoid re-validating previous revisions)
SKIP_REV_FILES = True

# Show detailed per-error breakdown? (can be verbose for many files)
VERBOSE = True

# ---------------------------------------------------------------------------
# Resolve input files
# ---------------------------------------------------------------------------
if MODE == "file":
    input_files = [Path(INPUT_FILE)]
elif MODE == "folder":
    input_files = sorted(Path(INPUT_FOLDER).glob("*.json"))
elif MODE == "list":
    input_files = [Path(p) for p in INPUT_LIST]
else:
    raise ValueError(f"Unknown MODE: {MODE!r}. Use 'file', 'folder', or 'list'.")

# Filter out _rev files if configured
if SKIP_REV_FILES:
    input_files = [f for f in input_files if not f.stem.endswith("_rev")]

# Filter to existing files
missing = [f for f in input_files if not f.exists()]
input_files = [f for f in input_files if f.exists()]

print(f"Mode: {MODE}")
print(f"Files to validate: {len(input_files)}")
for f in input_files:
    print(f"  {f.name}")
if missing:
    print(f"\nMissing files ({len(missing)}):")
    for f in missing:
        print(f"  {f}")

Mode: folder
Files to validate: 10
  rdls_hevl-bgdtmrwcities_chattogram.json
  rdls_hevl-bgdtmrwcities_coxsbazar.json
  rdls_hevl-ecutmrwcities_quito.json
  rdls_hevl-kentmrwcities_nairobi.json
  rdls_hevl-kentmrwcities_nakuru.json
  rdls_hevl-npltmrwcities_khokana.json
  rdls_hevl-npltmrwcities_rapti.json
  rdls_hevl-psetmrwcities_nablus.json
  rdls_hevl-turistmrwcities_istanbul.json
  rdls_hevl-tzatmrwcities_daressalaam.json


## Cell 3 — Load Schema & Codelists

In [14]:
# =============================================================================
# LOAD SCHEMA, BUILD CONTEXT, CREATE AUTO-FIXER
# =============================================================================

# Load the JSON schema
schema = load_rdls_schema(SCHEMA_PATH)
print(f"Schema loaded: {SCHEMA_PATH.name}")

# Load codelist reference from YAML
schema_cfg = load_yaml(SCHEMA_YAML)
closed_codelists = schema_cfg.get("codelists", {})
open_codelists = schema_cfg.get("open_codelists", {})
all_codelists = {**closed_codelists, **open_codelists}
print(f"Codelists: {len(closed_codelists)} closed, {len(open_codelists)} open")

# Build SchemaContext (enum lookup, field aliases, required fields, allowed props)
ctx = SchemaContext(schema)
print(f"SchemaContext built:")
print(f"  Enum fields: {len(ctx.enum_lookup)}")
print(f"  Field aliases: {len(ctx.field_aliases)}")
print(f"  Required-field defs: {len(ctx.required_lookup)}")
print(f"  Allowed-props contexts: {len(ctx.allowed_props)}")

# Load defaults and create auto-fixer
defaults = load_yaml(DEFAULTS_YAML)
fixer = AutoFixer(
    ctx, defaults,
    schema_gap_fields=defaults.get("schema_gap_fields", {}),
)
print(f"AutoFixer ready:")
print(f"  Hazard process defaults: {len(fixer.hazard_process_defaults)}")
print(f"  Schema gap fields: {list(fixer._schema_gap_fields.keys())}")


Schema loaded: rdls_schema_v0.3.json
Codelists: 24 closed, 5 open
SchemaContext built:
  Enum fields: 23
  Field aliases: 17
  Required-field defs: 19
  Allowed-props contexts: 26
AutoFixer ready:
  Hazard process defaults: 11
  Schema gap fields: ['Exposure_item', 'VulnerabilityFunction', 'FragilityFunction', 'DamageToLossFunction', 'EngineeringDemandFunction', 'SocioEconomicIndex', 'Metric']


## Cell 4 — Validation Engine

In [15]:
# =============================================================================
# VALIDATION ENGINE
# =============================================================================
# All validation functions are now in src.validate_qa:
#   - validate_against_schema(record, schema) -> list of error dicts
#   - categorize_error(err, path) -> category string
#   - check_business_rules(record) -> list of issue dicts
#
# Business rules checked:
#   1. Required attribution roles (publisher, creator, contact_point)
#   2. Resource must have download_url OR access_url
#   3. Entity must have email OR url
#   4. Schema link in links array
#   5. risk_data_type consistency with HEVL blocks
#   6. Spatial countries when scale=national/sub-national
#   7. Metric: quantity_kind=monetary requires currency
#   8. Loss impact_and_losses: quantity_kind=monetary requires currency
# =============================================================================

print("Validation engine loaded (from src.validate_qa).")


Validation engine loaded (from src.validate_qa).


## Cell 5 — Auto-Fix Engine

In [16]:
# =============================================================================
# AUTO-FIX ENGINE (Schema-Driven)
# =============================================================================
# The auto-fix engine is now in src.validate_qa.AutoFixer:
#   fixer = AutoFixer(ctx, defaults, schema_gap_fields)
#   fixed, fix_log = fixer.fix_record(record, errors, schema)
#
# 5 passes:
#   Pass 0: Structural repair (exposure object->array, hazard_process array->string)
#   Pass 1: Error-driven fixes (empty removal, type coercion, codelist correction)
#   Pass 2: Deep clean remaining empties in non-required fields
#   Pass 3: Structural inference (hazard events, socio_economic)
#   Pass 4: Additional properties cleanup (remove non-schema fields)
#
# Config-driven:
#   - hazard_process_defaults: from rdls_defaults.yaml
#   - schema_gap_fields: from rdls_defaults.yaml (e.g. Metric.currency, Exposure_item.id)
#   - fuzzy codelist matching: via SchemaContext.fuzzy_codelist_fix()
# =============================================================================

print(f"Auto-fix engine loaded (from src.validate_qa.AutoFixer).")
print(f"  Codelist correction: field-scoped fuzzy match against {len(ctx.enum_lookup)} enum fields")
print(f"  Field aliases: {len(ctx.field_aliases)} property->enum mappings")


Auto-fix engine loaded (from src.validate_qa.AutoFixer).
  Codelist correction: field-scoped fuzzy match against 23 enum fields
  Field aliases: 17 property->enum mappings


## Cell 6 — Run Validation

In [17]:
# =============================================================================
# RUN VALIDATION ON ALL INPUT FILES
# =============================================================================

all_results = []  # List of per-file result dicts

for filepath in input_files:
    print(f"\n{'='*80}")
    print(f"FILE: {filepath.name}")
    print(f"{'='*80}")
    
    # Load the file
    try:
        data = load_json(filepath)
    except Exception as e:
        print(f"  ERROR loading file: {e}")
        all_results.append({
            "file": filepath.name,
            "path": str(filepath),
            "load_error": str(e),
            "records": 0,
        })
        continue
    
    # Extract records (handle both wrapped and unwrapped formats)
    if isinstance(data, dict) and "datasets" in data:
        records = data["datasets"]
        is_wrapped = True
    elif isinstance(data, list):
        records = data
        is_wrapped = False
    elif isinstance(data, dict):
        # Single record, not wrapped
        records = [data]
        is_wrapped = False
    else:
        print(f"  ERROR: Unexpected JSON structure (type={type(data).__name__})")
        continue
    
    print(f"  Format: {'wrapped {datasets: [...]}' if is_wrapped else 'unwrapped'}")
    print(f"  Records: {len(records)}")
    
    file_result = {
        "file": filepath.name,
        "path": str(filepath),
        "records": len(records),
        "is_wrapped": is_wrapped,
        "record_results": [],
    }
    
    for idx, record in enumerate(records):
        ds_id = record.get("id", f"record_{idx}")
        print(f"\n  --- Record {idx}: {ds_id} ---")
        
        # 1. Schema validation
        schema_errors = validate_against_schema(record, schema)
        
        # 2. Business rules
        business_errors = check_business_rules(record)
        
        all_errors = schema_errors + business_errors
        
        # Count by category
        cat_counts = Counter(e["category"] for e in all_errors)
        
        is_valid = len(schema_errors) == 0
        print(f"  Schema valid: {is_valid}")
        print(f"  Schema errors: {len(schema_errors)}")
        print(f"  Business rule issues: {len(business_errors)}")
        print(f"  Total issues: {len(all_errors)}")
        
        if cat_counts:
            print(f"  By category:")
            for cat, cnt in cat_counts.most_common():
                print(f"    {cat}: {cnt}")
        
        if VERBOSE and all_errors:
            print(f"\n  Detailed errors:")
            for i, err in enumerate(all_errors):
                msg_short = err['message'][:120]
                print(f"    [{i+1:2d}] [{err['category']}] {err['path']}")
                print(f"         {msg_short}")
        
        record_result = {
            "id": ds_id,
            "is_valid": is_valid,
            "schema_errors": len(schema_errors),
            "business_issues": len(business_errors),
            "total_issues": len(all_errors),
            "categories": dict(cat_counts),
            "errors": all_errors,
            "record": record,
        }
        file_result["record_results"].append(record_result)
    
    all_results.append(file_result)

print(f"\n\n{'='*80}")
print(f"VALIDATION COMPLETE: {len(input_files)} files, "
      f"{sum(r['records'] for r in all_results)} records")
print(f"{'='*80}")


FILE: rdls_hevl-bgdtmrwcities_chattogram.json
  Format: wrapped {datasets: [...]}
  Records: 1

  --- Record 0: rdls_hevl-bgdtmrwcities_chattogram ---
  Schema valid: True
  Schema errors: 0
  Business rule issues: 0
  Total issues: 0

FILE: rdls_hevl-bgdtmrwcities_coxsbazar.json
  Format: wrapped {datasets: [...]}
  Records: 1

  --- Record 0: rdls_lss-tmrwcities_coxsbazar ---
  Schema valid: True
  Schema errors: 0
  Business rule issues: 0
  Total issues: 0

FILE: rdls_hevl-ecutmrwcities_quito.json
  Format: wrapped {datasets: [...]}
  Records: 1

  --- Record 0: rdls_hevl-ecutmrwcities_quito ---
  Schema valid: True
  Schema errors: 0
  Business rule issues: 0
  Total issues: 0

FILE: rdls_hevl-kentmrwcities_nairobi.json
  Format: wrapped {datasets: [...]}
  Records: 1

  --- Record 0: rdls_hevl-kentmrwcities_nairobi ---
  Schema valid: True
  Schema errors: 0
  Business rule issues: 0
  Total issues: 0

FILE: rdls_hevl-kentmrwcities_nakuru.json
  Format: wrapped {datasets: [...]}

## Cell 7 — Summary Table

In [7]:
# =============================================================================
# SUMMARY TABLE
# =============================================================================

print(f"{'File':<55} {'Records':>7} {'Valid':>6} {'Errors':>7} {'Categories'}")
print("-" * 110)

total_records = 0
total_valid = 0
total_errors = 0
global_categories = Counter()

for fr in all_results:
    if "load_error" in fr:
        print(f"{fr['file']:<55} {'LOAD ERROR':>7}  {fr['load_error'][:50]}")
        continue
    
    n_records = fr["records"]
    n_valid = sum(1 for rr in fr["record_results"] if rr["is_valid"])
    n_errors = sum(rr["total_issues"] for rr in fr["record_results"])
    cats = Counter()
    for rr in fr["record_results"]:
        cats.update(rr["categories"])
    
    total_records += n_records
    total_valid += n_valid
    total_errors += n_errors
    global_categories.update(cats)
    
    cat_str = ", ".join(f"{c}:{n}" for c, n in cats.most_common(5))
    valid_str = f"{n_valid}/{n_records}"
    print(f"{fr['file']:<55} {n_records:>7} {valid_str:>6} {n_errors:>7}  {cat_str}")

print("-" * 110)
valid_str = f"{total_valid}/{total_records}"
print(f"{'TOTAL':<55} {total_records:>7} {valid_str:>6} {total_errors:>7}")

if global_categories:
    print(f"\nGlobal error categories:")
    for cat, cnt in global_categories.most_common():
        print(f"  {cat}: {cnt}")

File                                                    Records  Valid  Errors Categories
--------------------------------------------------------------------------------------------------------------
rdls_hevl-bgdtmrwcities_chattogram.json                       1    0/1       6  invalid_codelist:6
rdls_hevl-bgdtmrwcities_coxsbazar.json                        1    0/1      12  invalid_codelist:10, empty_string:2
rdls_hevl-ecutmrwcities_quito.json                            1    1/1       0  
rdls_hevl-kentmrwcities_nairobi.json                          1    1/1       0  
rdls_hevl-kentmrwcities_nakuru.json                           1    1/1       0  
rdls_hevl-npltmrwcities_khokana.json                          1    1/1       0  
rdls_hevl-npltmrwcities_rapti.json                            1    1/1       0  
rdls_hevl-psetmrwcities_nablus.json                           1    1/1       0  
rdls_hevl-turistmrwcities_istanbul.json                       1    1/1       0  
rdls_hevl-tzatmrw

## Cell 8 — Apply Auto-Fixes & Re-Validate

In [8]:
# =============================================================================
# APPLY AUTO-FIXES AND RE-VALIDATE
# =============================================================================

revised_results = []  # List of per-file revised result dicts

for fr in all_results:
    if "load_error" in fr or not fr.get("record_results"):
        continue
    
    filepath = Path(fr["path"])
    print(f"\n{'='*80}")
    print(f"AUTO-FIX: {filepath.name}")
    print(f"{'='*80}")
    
    revised_records = []
    file_fix_log = []
    
    for rr in fr["record_results"]:
        ds_id = rr["id"]
        record = rr["record"]
        errors = rr["errors"]
        
        if rr["total_issues"] == 0:
            print(f"  {ds_id}: Already valid, no fixes needed.")
            revised_records.append(record)
            continue
        
        # Apply auto-fixes
        fixed_record, fix_log = fixer.fix_record(record, errors, schema)
        file_fix_log.extend(fix_log)
        
        # Count by severity
        sev_counts = Counter(f.get("severity", "auto") for f in fix_log)
        print(f"  {ds_id}: {len(fix_log)} fixes ({sev_counts.get('auto',0)} auto, "
              f"{sev_counts.get('review',0)} review)")
        
        # Show review-severity fixes prominently
        review_fixes = [f for f in fix_log if f.get("severity") == "review"]
        if review_fixes:
            print(f"  *** NEEDS USER REVIEW ({len(review_fixes)} items):")
            for fix in review_fixes:
                print(f"    [{fix['action']}] {fix['path']}: {str(fix['old'])[:30]} -> {str(fix['new'])[:40]}")
        
        # Re-validate
        post_errors = validate_against_schema(fixed_record, schema)
        post_business = check_business_rules(fixed_record)
        
        print(f"  Post-fix: schema_errors={len(post_errors)}, business_issues={len(post_business)}")
        
        if post_errors:
            print(f"  Remaining schema errors:")
            for i, err in enumerate(post_errors[:10]):
                print(f"    [{i+1}] [{err['category']}] {err['path']}: {err['message'][:100]}")
            if len(post_errors) > 10:
                print(f"    ... and {len(post_errors) - 10} more")
        
        revised_records.append(fixed_record)
    
    revised_results.append({
        "file": fr["file"],
        "path": fr["path"],
        "is_wrapped": fr["is_wrapped"],
        "revised_records": revised_records,
        "fix_count": len(file_fix_log),
        "fix_log": file_fix_log,
    })

print(f"\n\nAuto-fix complete. {sum(r['fix_count'] for r in revised_results)} total fixes applied.")


AUTO-FIX: rdls_hevl-bgdtmrwcities_chattogram.json
  rdls_hevl-bgdtmrwcities_chattogram: 0 fixes (0 auto, 0 review)
  Post-fix: schema_errors=6, business_issues=0
  Remaining schema errors:
    [1] [invalid_codelist] exposure.0.metrics.2.dimension: 'area' is not one of ['structure', 'content', 'product', 'disruption', 'population', 'index']
    [2] [invalid_codelist] exposure.2.metrics.0.dimension: 'length' is not one of ['structure', 'content', 'product', 'disruption', 'population', 'index']
    [3] [invalid_codelist] exposure.3.category: 'indicators' is not one of ['agriculture', 'buildings', 'infrastructure', 'population', 'natural_env
    [4] [invalid_codelist] exposure.3.metrics.0.dimension: 'area' is not one of ['structure', 'content', 'product', 'disruption', 'population', 'index']
    [5] [invalid_codelist] loss.losses.2.impact_and_losses.impact_metric: 'affected_population' is not one of ['damage_ratio', 'mean_damage_ratio', 'probability', 'damage_ind
    [6] [invalid_codelist

## Cell 9 — Save Revised Files

In [9]:
# =============================================================================
# SAVE REVISED FILES
# =============================================================================

if not SAVE_REVISED:
    print("SAVE_REVISED is False — skipping.")
else:
    saved_files = []
    
    for rr in revised_results:
        if rr["fix_count"] == 0:
            print(f"  {rr['file']}: No fixes applied, skipping save.")
            continue
        
        orig_path = Path(rr["path"])
        
        # Determine output path
        if REVISED_OUTPUT_DIR:
            out_dir = Path(REVISED_OUTPUT_DIR)
        else:
            out_dir = orig_path.parent
        
        # Create _rev filename
        stem = orig_path.stem
        rev_name = f"{stem}_rev.json"
        rev_path = out_dir / rev_name
        
        # Reconstruct the full structure
        if rr["is_wrapped"]:
            output_data = {"datasets": rr["revised_records"]}
        elif len(rr["revised_records"]) == 1:
            output_data = rr["revised_records"][0]
        else:
            output_data = rr["revised_records"]
        
        # If wrapped, keep same format
        if rr["is_wrapped"]:
            output_data = {"datasets": rr["revised_records"]}
        
        write_json(rev_path, output_data)
        saved_files.append(rev_path)
        print(f"  Saved: {rev_path}")
        print(f"         ({rr['fix_count']} fixes applied to {rr['file']})")
    
    print(f"\n{len(saved_files)} revised file(s) saved.")

  rdls_hevl-bgdtmrwcities_chattogram.json: No fixes applied, skipping save.
  rdls_hevl-bgdtmrwcities_coxsbazar.json: No fixes applied, skipping save.
  rdls_hevl-ecutmrwcities_quito.json: No fixes applied, skipping save.
  rdls_hevl-kentmrwcities_nairobi.json: No fixes applied, skipping save.
  rdls_hevl-kentmrwcities_nakuru.json: No fixes applied, skipping save.
  rdls_hevl-npltmrwcities_khokana.json: No fixes applied, skipping save.
  rdls_hevl-npltmrwcities_rapti.json: No fixes applied, skipping save.
  rdls_hevl-psetmrwcities_nablus.json: No fixes applied, skipping save.
  rdls_hevl-turistmrwcities_istanbul.json: No fixes applied, skipping save.
  rdls_hevl-tzatmrwcities_daressalaam.json: No fixes applied, skipping save.

0 revised file(s) saved.


## Cell 10 — Final Validation of Revised Files

In [10]:
# =============================================================================
# FINAL VALIDATION OF REVISED FILES
# =============================================================================

if not SAVE_REVISED or not saved_files:
    print("No revised files to validate.")
else:
    print(f"\nRe-validating {len(saved_files)} revised file(s)...\n")
    
    for rev_path in saved_files:
        print(f"{'='*80}")
        print(f"REVISED: {rev_path.name}")
        print(f"{'='*80}")
        
        data = load_json(rev_path)
        if isinstance(data, dict) and "datasets" in data:
            records = data["datasets"]
        elif isinstance(data, list):
            records = data
        else:
            records = [data]
        
        for idx, record in enumerate(records):
            ds_id = record.get("id", f"record_{idx}")
            
            schema_errors = validate_against_schema(record, schema)
            business_errors = check_business_rules(record)
            
            is_valid = len(schema_errors) == 0
            status = "VALID" if is_valid else "INVALID"
            print(f"  {ds_id}: {status} (schema_errors={len(schema_errors)}, business_issues={len(business_errors)})")
            
            if schema_errors:
                cat_counts = Counter(e["category"] for e in schema_errors)
                print(f"  Remaining errors by category: {dict(cat_counts)}")
                for i, err in enumerate(schema_errors):
                    print(f"    [{i+1}] [{err['category']}] {err['path']}: {err['message'][:120]}")
            
            if business_errors:
                for i, err in enumerate(business_errors):
                    print(f"    [BR{i+1}] [{err['category']}] {err['path']}: {err['message'][:120]}")
    
    print(f"\nFinal validation complete.")

No revised files to validate.


## Cell 11 — Comprehensive Summary Report

In [11]:
# =============================================================================
# COMPREHENSIVE SUMMARY REPORT
# =============================================================================
# This report provides:
#   1. Per-file comparison table (original vs revised errors)
#   2. Detailed revision log grouped by fix type
#   3. MANDATORY FIELDS AUTO-FILLED section (requires user review)
#   4. Remaining unfixed errors (manual review needed)
#   5. Global statistics
# =============================================================================

from datetime import datetime

print("=" * 100)
print(f"  RDLS v0.3 METADATA VALIDATION & REVISION REPORT")
print(f"  Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Schema: {SCHEMA_PATH.name}")
print(f"  Files: {len(input_files)} | Mode: {MODE}")
print("=" * 100)

# -------------------------------------------------------------------------
# Section 1: Per-File Comparison Table
# -------------------------------------------------------------------------
print(f"\n{'─'*100}")
print(f"  SECTION 1: VALIDATION SUMMARY (Original → Revised)")
print(f"{'─'*100}")
print(f"  {'File':<48} {'Orig':>5} {'Rev':>5} {'Fixed':>6} {'Rate':>6} {'Review':>7} {'Status'}")
print(f"  {'-'*48} {'-'*5} {'-'*5} {'-'*6} {'-'*6} {'-'*7} {'-'*8}")

grand_orig = 0
grand_rev = 0
grand_fixed = 0
grand_review = 0

for fr, rr in zip(all_results, revised_results):
    if "load_error" in fr:
        print(f"  {fr['file']:<48} {'LOAD ERROR':>5}")
        continue
    
    for orig_rr, rev_rec in zip(fr["record_results"], rr["revised_records"]):
        # Re-validate revised
        rev_schema_errs = validate_against_schema(rev_rec, schema)
        rev_biz_errs = check_business_rules(rev_rec)
        
        orig_total = orig_rr["total_issues"]
        rev_total = len(rev_schema_errs) + len(rev_biz_errs)
        fixed = orig_total - rev_total
        pct = f"{(fixed / orig_total * 100):.0f}%" if orig_total > 0 else "N/A"
        
        review_count = sum(1 for f in rr["fix_log"] if f.get("severity") == "review")
        status = "✓ VALID" if len(rev_schema_errs) == 0 else "✗ INVALID"
        
        grand_orig += orig_total
        grand_rev += rev_total
        grand_fixed += fixed
        grand_review += review_count
        
        print(f"  {fr['file']:<48} {orig_total:>5} {rev_total:>5} {fixed:>6} {pct:>6} {review_count:>7} {status}")

print(f"  {'-'*48} {'-'*5} {'-'*5} {'-'*6} {'-'*6} {'-'*7} {'-'*8}")
grand_pct = f"{(grand_fixed / grand_orig * 100):.0f}%" if grand_orig > 0 else "N/A"
print(f"  {'TOTAL':<48} {grand_orig:>5} {grand_rev:>5} {grand_fixed:>6} {grand_pct:>6} {grand_review:>7}")

# -------------------------------------------------------------------------
# Section 2: Detailed Revision Log by Fix Type
# -------------------------------------------------------------------------
print(f"\n{'─'*100}")
print(f"  SECTION 2: REVISION LOG (What was changed)")
print(f"{'─'*100}")

# Aggregate all fixes by action type
all_fixes = []
for rr in revised_results:
    for fix in rr["fix_log"]:
        all_fixes.append({**fix, "file": rr["file"]})

action_groups = defaultdict(list)
for fix in all_fixes:
    action_groups[fix["action"]].append(fix)

for action, fixes in sorted(action_groups.items(), key=lambda x: -len(x[1])):
    sev = fixes[0].get("severity", "auto")
    sev_icon = "⚠" if sev == "review" else "✓"
    print(f"\n  {sev_icon} {action} ({len(fixes)} occurrences) [{sev.upper()}]")
    
    # Show unique old→new mappings
    unique_changes = {}
    for f in fixes:
        key = f"{f['old']} → {f['new']}"
        if key not in unique_changes:
            unique_changes[key] = []
        unique_changes[key].append(f)
    
    for change, instances in list(unique_changes.items())[:5]:
        files = set(f["file"] for f in instances)
        files_str = ", ".join(sorted(files)[:3])
        if len(files) > 3:
            files_str += f" (+{len(files)-3} more)"
        # Truncate long change descriptions
        change_display = change[:80] + "..." if len(change) > 80 else change
        print(f"    {change_display}")
        print(f"      Files: {files_str} ({len(instances)}x)")
    if len(unique_changes) > 5:
        print(f"    ... and {len(unique_changes)-5} more unique changes")

# -------------------------------------------------------------------------
# Section 3: MANDATORY FIELDS AUTO-FILLED (User Review Required!)
# -------------------------------------------------------------------------
review_fixes = [f for f in all_fixes if f.get("severity") == "review"]
if review_fixes:
    print(f"\n{'─'*100}")
    print(f"  SECTION 3: ⚠ MANDATORY FIELDS AUTO-FILLED — USER REVIEW REQUIRED")
    print(f"{'─'*100}")
    print(f"  The following {len(review_fixes)} mandatory field(s) were empty or missing.")
    print(f"  They were filled automatically using data from WITHIN the same record")
    print(f"  (parent/sibling context or default mappings). NEVER from external sources.")
    print(f"  Please verify each value is correct for your dataset.\n")
    
    for i, fix in enumerate(review_fixes):
        print(f"  [{i+1:3d}] File: {fix['file']}")
        print(f"        Path: {fix['path']}")
        print(f"        Action: {fix['action']}")
        print(f"        Old value: {str(fix['old'])[:60]}")
        print(f"        New value: {str(fix['new'])[:60]}")
        print()
else:
    print(f"\n{'─'*100}")
    print(f"  SECTION 3: No mandatory fields were auto-filled.")
    print(f"{'─'*100}")

# -------------------------------------------------------------------------
# Section 4: Remaining Unfixed Errors (Manual Review Needed)
# -------------------------------------------------------------------------
print(f"\n{'─'*100}")
print(f"  SECTION 4: REMAINING ERRORS — MANUAL REVIEW REQUIRED")
print(f"{'─'*100}")

any_remaining = False
for fr, rr in zip(all_results, revised_results):
    if "load_error" in fr:
        continue
    for orig_rr, rev_rec in zip(fr["record_results"], rr["revised_records"]):
        rev_schema_errs = validate_against_schema(rev_rec, schema)
        rev_biz_errs = check_business_rules(rev_rec)
        
        if rev_schema_errs or rev_biz_errs:
            any_remaining = True
            print(f"\n  File: {fr['file']} ({orig_rr['id']})")
            
            if rev_schema_errs:
                # Group remaining errors by category
                cat_errs = defaultdict(list)
                for err in rev_schema_errs:
                    cat_errs[err["category"]].append(err)
                
                for cat, errs in sorted(cat_errs.items(), key=lambda x: -len(x[1])):
                    print(f"    [{cat}] ({len(errs)} errors)")
                    for err in errs[:5]:
                        msg_short = err["message"][:90]
                        print(f"      {err['path']}: {msg_short}")
                    if len(errs) > 5:
                        print(f"      ... and {len(errs)-5} more")
            
            if rev_biz_errs:
                print(f"    [business_rules] ({len(rev_biz_errs)} issues)")
                for err in rev_biz_errs:
                    print(f"      {err['path']}: {err['message'][:90]}")

if not any_remaining:
    print(f"  All files are now schema-valid! No remaining errors.")

# -------------------------------------------------------------------------
# Section 5: Global Statistics
# -------------------------------------------------------------------------
print(f"\n{'─'*100}")
print(f"  SECTION 5: GLOBAL STATISTICS")
print(f"{'─'*100}")

total_auto = sum(1 for f in all_fixes if f.get("severity") == "auto")
total_review = sum(1 for f in all_fixes if f.get("severity") == "review")

print(f"  Files processed:       {len(input_files)}")
print(f"  Total original errors: {grand_orig}")
print(f"  Total fixed:           {grand_fixed} ({grand_pct})")
print(f"  Total remaining:       {grand_rev}")
print(f"  Fixes breakdown:")
print(f"    Auto (safe):         {total_auto}")
print(f"    Review (check!):     {total_review}")
print(f"  Fix types:")

action_counts = Counter(f["action"] for f in all_fixes)
for action, count in action_counts.most_common():
    sev = next((f.get("severity", "auto") for f in all_fixes if f["action"] == action), "auto")
    marker = " ⚠" if sev == "review" else ""
    print(f"    {action}: {count}{marker}")

print(f"\n{'='*100}")
print(f"  END OF REPORT")
print(f"{'='*100}")

  RDLS v0.3 METADATA VALIDATION & REVISION REPORT
  Generated: 2026-02-16 10:08:01
  Schema: rdls_schema_v0.3.json
  Files: 10 | Mode: folder

────────────────────────────────────────────────────────────────────────────────────────────────────
  SECTION 1: VALIDATION SUMMARY (Original → Revised)
────────────────────────────────────────────────────────────────────────────────────────────────────
  File                                              Orig   Rev  Fixed   Rate  Review Status
  ------------------------------------------------ ----- ----- ------ ------ ------- --------
  rdls_hevl-bgdtmrwcities_chattogram.json              6     6      0     0%       0 ✗ INVALID
  rdls_hevl-bgdtmrwcities_coxsbazar.json              12    12      0     0%       0 ✗ INVALID
  rdls_hevl-ecutmrwcities_quito.json                   0     0      0    N/A       0 ✓ VALID
  rdls_hevl-kentmrwcities_nairobi.json                 0     0      0    N/A       0 ✓ VALID
  rdls_hevl-kentmrwcities_nakuru.json   

---

## Design Notes

### Empty field handling rules
| Field type | Empty value | Action | Severity |
|:-----------|:-----------|:-------|:---------|
| **Optional** | `""`, `{}`, `[]` | **Remove** the field | `auto` |
| **Mandatory** | `""`, `{}`, `[]`, missing | **Infer from context** within the same record | `review` |
| **Mandatory** (can't infer) | missing | **Leave** — flag in report for manual review | `manual` |

**Context sources for mandatory field inference** (never from internet):
1. Parent object (e.g. `event_set.calculation_method` -> `event.calculation_method`)
2. Sibling/child objects (e.g. child events -> parent `event_set.hazards[].hazard_process`)
3. Hazard type -> default process type mapping (e.g. `flood` -> `fluvial_flood`)
4. Schema minimum values (e.g. `reference_year` -> `1900`)

### Schema-driven approach
- **All codelist lookups** extracted dynamically from the JSON schema (`ENUM_LOOKUP`)
- **Field alias resolution** maps JSON property names to `$defs` enum keys via `FIELD_ALIASES`
  - e.g. `dimension` -> `metric_dimension`, `relationship` -> `relationship_type`, `category` -> `exposure_category`
  - Built automatically by scanning all `$defs` properties for `$ref` pointers
- **Codelist correction** uses fuzzy matching (`difflib.get_close_matches`) against resolved schema enums
- **No hardcoded error patterns** — works with any valid or invalid RDLS record

### 5-pass auto-fix engine

**Pass 0 — Structural repair (fix wrong JSON types)**
- `exposure`: object `{categories: [...]}` -> array `[...]` (restructure)
- `hazard_processes` (plural): rename to `hazard_process` (singular) + unwrap array to string
- `hazard_process`: array `["value"]` -> string `"value"` (unwrap single-element)
- `hazard_process`: `{}` (empty object) -> `""` (empty string, for Pass 3 inference)
- Dot-path keys (e.g. `occurrence.deterministic.thresholds`) -> removed
- After structural repair, errors are **re-validated** so subsequent passes work on correct structure

**Pass 1 — Error-driven fixes (empty removal, type coercion, codelist correction)**
- Empty strings `""` in optional fields -> field removed
- Empty objects `{}` in optional fields -> field removed
- Empty arrays `[]` in optional fields -> field removed
- String numbers where schema expects numeric -> type coerced (`"2"` -> `2`, `"300"` -> `300`)
- Invalid codelist values -> fuzzy-matched to closest valid value from schema

**Pass 2 — Deep clean remaining empties in non-required fields**

**Pass 3 — Structural inference** — all flagged `severity=review`
- `hazard_process` missing -> inferred from child events, or default from hazard_type
- `event.calculation_method` missing -> inherited from parent `event_set`
- `event.hazard` missing -> copied from parent `event_set.hazards[0]`
- `event.occurrence` missing -> placeholder based on `event_set.analysis_type`
- `reference_year` missing -> schema minimum (1900)

**Pass 4 — Additional properties cleanup (schema compliance)**
- Walks entire record structure, resolves which `$defs` type applies at each level
- Removes any field NOT defined in that type's schema `properties`
- `SCHEMA_ALLOWED_PROPS`: built from schema `$defs`, keyed by type name
- **Schema gap preservation**: `id` field in `Exposure_item`, `VulnerabilityFunction`,
  `FragilityFunction`, `SocioEconomicIndex` is preserved (used by ALL originals but not yet in schema)
- Fields cleaned by Pass 4 include:
  - `hazard_processes` (plural) -> leftover after `hazard_process` (singular) added
  - `currency` in `Metric` objects -> only allowed in `impact_and_losses`
  - `exp_item` in `Metric` objects -> not a schema field
  - `disaster_identifiers` in `Hazard` objects -> not a schema field
  - `thresholds` in `occurrence.deterministic` -> not a schema field

### Schema compliance verification
After revision, each `_rev.json` is verified with 4 checks:
1. **JSON Schema validation** — Draft 2020-12 validation against `rdls_schema_v0.3.json`
2. **No additional properties** — every field must exist in its context's schema definition
3. **No structural violations** — correct JSON types (array/string/object) at every path
4. **Added fields audit** — original vs revised diff shows only legitimate Pass 3 inferences

**Compliance results (v3, with Pass 0-4):**
- Extra properties: **0** across all 10 files (CLEAN)
- Structural violations: **0** across all 10 files (CLEAN)
- Added fields: **44** total — ALL from Pass 3 inference (hazard_process, occurrence, calculation_method)

Known schema gaps (preserved, pending schema update):
- `id` in `Exposure_item`: used by all examples, not in schema
- `id` in `VulnerabilityFunction`, `FragilityFunction`, `SocioEconomicIndex`: same

### Summary report sections
1. **Per-file comparison table** — original vs revised error counts, fix rate, status
2. **Revision log** — grouped by fix type with unique old->new mappings
3. **Mandatory fields auto-filled** — every auto-filled required field listed for user review
4. **Remaining errors** — unfixable issues requiring manual intervention
5. **Global statistics** — totals, breakdown by severity and fix type

### Safety: No cross-field contamination
- Fuzzy matcher is **field-scoped only** — only considers the specific field's enum values
- `FIELD_ALIASES` resolves property names to their correct `$defs` enum keys

### Test results (10 example files, 5-pass engine v3)
| File | Original | Post-fix | Rate | Pass 0 repairs | Notes |
|:-----|:---------|:---------|:-----|:---------------|:------|
| ESA landcover (exposure) | 10 | 0 | 100% | 0 | VALID |
| NISMOD Africa transport (exposure) | 69 | 0 | 100% | 0 | VALID |
| Aruba ICRA (hazard+exposure) | 65 | 25 | 62% | 20 (exposure restructure + hazard_processes) | codelist + empty strings |
| Freetown flood risk (HEL) | 43 | 5 | 88% | 0 | 5 unfixable codelists |
| Chattogram HEVL | 72 | 6 | 92% | 1 (hazard_process {}) | wrong dimension/category |
| Cox's Bazar HEVL | 119 | 12 | 90% | 8 (hazard_processes) | wrong codelist + empty triggers |
| Gambia flood+coast (HEVL) | 162 | 9 | 94% | 17 (hazard_process arrays + dot-paths) | empty triggers + intensity_measure |
| Dar es Salaam HEVL | 72 | 10 | 86% | 3 (hazard_processes) | wrong codelist + empty triggers |
| Freetown flood hazard | 12 | 12 | 0% | 2 (dot-paths) | `"website"` not in access_modality |
| JRC windstorm (loss) | 8 | 0 | 100% | 0 | VALID |
| **TOTAL** | **632** | **79** | **87.5%** | **51** | 3 files VALID, 0 extra props, 0 structural |

### Improvement from Pass 0 structural repair
- Previous (Pass 1-3 only): 631 -> 103 errors (84%)
- **Current (Pass 0-4): 632 -> 79 errors (87.5%)**
- **Gambia**: 162 -> 37 (was) -> **162 -> 9** (now) = 28 fewer errors from array->string unwrapping
- **ICRA**: exposure restructured from `{categories:[...]}` to `[...]`, enabling proper validation
- **All hazard_processes (plural)**: renamed to `hazard_process` (singular) in ICRA/CoxsBazar/DarEsSalaam